### Manual: 05-clone-pin

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Clone MIT BEVFusion and check out the pinned commit `da93b27`.

Also records the static evidence for the pin: the release commit has neither
`add_depth_features` nor `height_expand`, its depth tensor is built as
`torch.zeros(batch_size, 6, 1, *image_size)` and `dtransform` starts with
`Conv2d(1, 8, 1)`.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "clone.log")
report = {"repo": sc.REPO, "pin_short": sc.PIN_SHORT}

if not os.path.isdir(os.path.join(sc.REPO, ".git")):
    rc, _ = sc.sh('git clone --quiet "%s" "%s"' % (sc.REPO_URL, sc.REPO), log=log, timeout=1800)
    report["clone_rc"] = rc
else:
    report["clone_rc"] = "skipped (already cloned)"

rc, _ = sc.sh('git -C "%s" checkout --quiet %s' % (sc.REPO, sc.PIN_SHORT), log=log)
report["checkout_rc"] = rc
report["head"] = sc.capture('git -C "%s" rev-parse HEAD' % sc.REPO)
report["head_short"] = report["head"][:7]
report["log1"] = sc.capture('git -C "%s" log -1 --format="%%H %%ci %%s"' % sc.REPO)
report["git_status"] = sc.capture('git -C "%s" status --porcelain' % sc.REPO)
report["branch"] = sc.capture('git -C "%s" rev-parse --abbrev-ref HEAD' % sc.REPO)

base_py = os.path.join(sc.REPO, "mmdet3d", "models", "vtransforms", "base.py")
depth_py = os.path.join(sc.REPO, "mmdet3d", "models", "vtransforms", "depth_lss.py")
report["depth_tensor_source"] = sc.capture('grep -n "torch.zeros(batch_size, 6, 1" "%s"' % base_py)
report["dtransform_source"] = sc.capture('grep -n "Conv2d(1, 8, 1)" "%s"' % depth_py)
report["add_depth_features_hits"] = int(
    sc.capture('grep -rn "add_depth_features" "%s" | wc -l' % sc.REPO) or 0
)
report["height_expand_hits"] = int(
    sc.capture('grep -rn "height_expand" "%s" | wc -l' % sc.REPO) or 0
)
report["config_exists"] = os.path.isfile(sc.CONFIG_PATH)

report["ok"] = bool(
    report["head"].startswith(sc.PIN_SHORT)
    and report["git_status"] == ""
    and report["add_depth_features_hits"] == 0
    and report["height_expand_hits"] == 0
    and report["config_exists"]
)
print(json.dumps(report, indent=2, sort_keys=True))
sc.save_json("pin.json", report)
sc.update(pin=report)
print("PIN_OK" if report["ok"] else "PIN_FAILED")
